# 05 - XLM-R: 3 seeds + control experiment (run on Google Colab)

**Before running:** Runtime -> Change runtime type -> **T4 GPU** -> Save.

**Why this notebook:** In Week 4 every number came from a single run. Here every experiment is repeated with **3 random seeds**, so we can report **mean +/- standard deviation**. We also add a **control experiment**:

| Condition | What it tells us |
|---|---|
| RU only | baseline |
| EN+IT+RU | does foreign data help? |
| EN+IT with **shuffled labels** + RU | same amount of text and training steps, but the sentiment labels are meaningless. If this does NOT help but EN+IT+RU does, the gain comes from real sentiment knowledge in English/Italian, not just from extra data. |

Training is also made more stable: warm-up, and small datasets get more epochs (at least about 150 update steps).

**Time:** roughly 45 to 70 minutes in total. Results are saved after every run, so if Colab disconnects you can re-run the cells and it will **skip finished runs** (as long as the Colab session was not reset).

Run cells top to bottom.

## 1. Install and check GPU

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn
import torch
print('GPU available:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - change runtime type to T4 GPU')

## 2. Upload data files
Select exactly these 3 files from `data\raw`: **`english_train.csv`, `italian_train.csv`, `roman_urdu.csv`** (note: *train*, not *test*).

In [ ]:
import os
from google.colab import files
if not all(os.path.exists(f) for f in ['english_train.csv', 'italian_train.csv', 'roman_urdu.csv']):
    uploaded = files.upload()
needed = ['english_train.csv', 'italian_train.csv', 'roman_urdu.csv']
missing = [f for f in needed if not os.path.exists(f)]
assert not missing, f'Missing files: {missing}. Upload them again (use italian_train.csv, not italian_test.csv).'
print('All files present.')

## 3. Load data (same cleaning and same train/test split as Weeks 2 to 4)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

LABEL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_TO_NAME = {0: 'positive', 1: 'neutral', 2: 'negative'}   # English/Italian files use 0/1/2

def load_multilingual(path):
    df = pd.read_csv(path)
    if pd.api.types.is_numeric_dtype(df['label']):
        df['label'] = df['label'].map(NUM_TO_NAME)
    df['text'] = df['text'].astype(str)
    return df[['text', 'label']].dropna().drop_duplicates(subset='text')

en = load_multilingual('english_train.csv')
it = load_multilingual('italian_train.csv')

ru = pd.read_csv('roman_urdu.csv', header=None, usecols=[0, 1],
                 names=['text', 'label'], on_bad_lines='skip').dropna(subset=['text'])
ru['label'] = ru['label'].str.strip().str.lower().replace({'neative': 'negative'})
ru['text'] = ru['text'].astype(str)
ru = ru[ru['text'].str.strip() != ''].drop_duplicates(subset='text')

# The test set is FIXED (same as before). Only training subsets and model seeds change.
ru_train, ru_test = train_test_split(ru, test_size=0.2, stratify=ru['label'], random_state=42)
foreign = pd.concat([en, it])
print('English:', len(en), ' Italian:', len(it), ' RU train:', len(ru_train), ' RU test:', len(ru_test))

## 4. Training helper

In [ ]:
import gc, math, time
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding, set_seed)

MODEL = 'xlm-roberta-base'
BATCH = 32
MIN_STEPS = 150          # small datasets get more epochs so training does not collapse
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def to_dataset(df):
    ds = Dataset.from_dict({'text': df['text'].tolist(),
                            'label': [LABEL2ID[l] for l in df['label']]})
    return ds.map(lambda b: tokenizer(b['text'], truncation=True, max_length=128),
                  batched=True, remove_columns=['text'])

test_ds = to_dataset(ru_test)
gold = [LABEL2ID[l] for l in ru_test['label']]

def auto_epochs(n_rows):
    steps_per_epoch = math.ceil(n_rows / BATCH)
    return int(min(30, max(3, math.ceil(MIN_STEPS / steps_per_epoch))))

def run(condition, ru_n, train_df, seed):
    set_seed(seed)
    t0 = time.time()
    epochs = auto_epochs(len(train_df))
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID)
    args = TrainingArguments(
        output_dir='out', per_device_train_batch_size=BATCH, per_device_eval_batch_size=64,
        num_train_epochs=epochs, learning_rate=2e-5, weight_decay=0.01, warmup_ratio=0.1,
        fp16=torch.cuda.is_available(), save_strategy='no', report_to='none',
        logging_steps=500, seed=seed)
    trainer = Trainer(model=model, args=args, train_dataset=to_dataset(train_df),
                      data_collator=DataCollatorWithPadding(tokenizer))
    trainer.train()
    pred = trainer.predict(test_ds).predictions.argmax(axis=-1)
    row = {'condition': condition, 'ru_n': ru_n, 'seed': seed, 'train_rows': len(train_df),
           'epochs': epochs,
           'accuracy': round(accuracy_score(gold, pred), 3),
           'macro_f1': round(f1_score(gold, pred, average='macro'), 3),
           'classes_predicted': int(len(set(pred.tolist()))),
           'minutes': round((time.time() - t0) / 60, 1)}
    print(row)
    del model, trainer
    gc.collect(); torch.cuda.empty_cache()
    return row

def shuffled_labels(df, seed):
    out = df.copy()
    out['label'] = np.random.default_rng(seed).permutation(out['label'].values)
    return out

## 5. Run all experiments (3 seeds)
Results are saved to `xlmr_seeds_results.csv` after every single run. If the cell stops (Colab disconnect), just run it again: finished runs are skipped.

In [ ]:
RESULTS_FILE = 'xlmr_seeds_results.csv'
SEEDS = [42, 43, 44]
SIZES = [100, 500, 2000]

results = pd.read_csv(RESULTS_FILE).to_dict('records') if os.path.exists(RESULTS_FILE) else []
done = {(r['condition'], r['ru_n'], r['seed']) for r in results}

def maybe_run(condition, ru_n, train_df, seed):
    if (condition, ru_n, seed) in done:
        print('skip (already done):', condition, ru_n, seed)
        return
    row = run(condition, ru_n, train_df, seed)
    results.append(row)
    done.add((condition, ru_n, seed))
    pd.DataFrame(results).to_csv(RESULTS_FILE, index=False)

for seed in SEEDS:
    print(f'\n===== SEED {seed} =====')
    maybe_run('zero-shot EN+IT', 0, foreign, seed)
    maybe_run('RU only', len(ru_train), ru_train, seed)
    for n in SIZES:
        sample = ru_train.sample(n=n, random_state=seed)       # a different subset for each seed
        maybe_run('RU only', n, sample, seed)
        maybe_run('EN+IT + RU', n, pd.concat([foreign, sample]), seed)
        maybe_run('EN+IT shuffled labels + RU', n,
                  pd.concat([shuffled_labels(foreign, seed), sample]), seed)

print('\nAll done. Runs saved:', len(results))

## 6. Summary: mean and standard deviation over 3 seeds
`collapsed_runs` = how many runs predicted only ONE class (a failed training run).

In [ ]:
res = pd.read_csv(RESULTS_FILE)
res['collapsed'] = res['classes_predicted'] <= 1

summary = (res.groupby(['condition', 'ru_n'])
              .agg(runs=('seed', 'count'),
                   f1_mean=('macro_f1', 'mean'), f1_std=('macro_f1', 'std'),
                   f1_min=('macro_f1', 'min'), f1_max=('macro_f1', 'max'),
                   acc_mean=('accuracy', 'mean'),
                   collapsed_runs=('collapsed', 'sum'))
              .round(3).reset_index()
              .sort_values(['ru_n', 'condition']))
summary.to_csv('xlmr_seeds_summary.csv', index=False)
summary

## 7. Graph (few-shot, with error bars)
TF-IDF reference lines are taken from Weeks 2 and 3 (full data 0.653).

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
styles = {'RU only': 'o-', 'EN+IT + RU': 's-', 'EN+IT shuffled labels + RU': '^--'}
for cond, style in styles.items():
    d = summary[(summary['condition'] == cond) & (summary['ru_n'].isin(SIZES))].sort_values('ru_n')
    ax.errorbar(d['ru_n'], d['f1_mean'], yerr=d['f1_std'].fillna(0), fmt=style, capsize=4, label=cond)

full = summary[(summary['condition'] == 'RU only') & (summary['ru_n'] == len(ru_train))]
if len(full):
    ax.axhline(full['f1_mean'].iloc[0], color='gray', ls=':', label='XLM-R, full Roman Urdu data')
ax.axhline(0.653, color='red', ls=':', alpha=0.6, label='TF-IDF, full Roman Urdu data (0.653)')
ax.axhline(1/3, color='black', ls=':', alpha=0.4, label='random guess (0.33)')
ax.set_xscale('log'); ax.set_xticks(SIZES); ax.set_xticklabels(SIZES)
ax.set_xlabel('Roman Urdu training examples (log scale)')
ax.set_ylabel('Macro-F1 on Roman Urdu test set')
ax.set_title('XLM-R few-shot: mean +/- std over 3 seeds')
ax.grid(alpha=0.3); ax.legend(fontsize=8); plt.tight_layout()
plt.savefig('xlmr_seeds_curve.png', dpi=150)
plt.show()

## 8. Download results
Put all 3 files in your project's `docs/` folder, then `git add .`, `git commit`, `git push`.

In [ ]:
files.download('xlmr_seeds_results.csv')
files.download('xlmr_seeds_summary.csv')
files.download('xlmr_seeds_curve.png')

## My notes (write in your own words)
1. **How big is the seed-to-seed variation (std)?** 
2. **Does EN+IT + RU still beat RU only after averaging 3 seeds?** 
3. **Control: does EN+IT with shuffled labels help? What does that say about where the gain comes from?** 
4. **Did any run collapse (predict only one class)?** 
5. **Zero-shot (EN+IT only): mean +/- std vs TF-IDF (0.345):** 